# Qwen2.5-1.5B-Instruct Reasoning 两阶段微调（Google Colab 版）

**支持平台**：Google Colab 免费版（T4 15 GB）/ Pro 版（A100 40 GB）均可运行

**流程（按顺序逐 Cell 执行）**：
1. 安装依赖
2. 挂载 Google Drive（可选，持久化保存产物）
3. 克隆项目 & 创建目录
4. 配置 HuggingFace Token
5. GPU 检查 & 自动适配超参
6. 数据集下载 & 本地缓存
7. ⚡ Baseline 评测（可选，用于对比）
8. ⚡ SFT 监督微调
9. ⚡ DPO 偏好优化
10. 合并 LoRA 权重
11. 推理对比
12. 最终评测 & 对比表

> **T4（免费）**：轻量训练约 40 分钟；**A100（Pro）**：全量训练约 30 分钟  
> ⚠️  免费 Colab 会话结束后 `/content` 数据丢失，**强烈建议挂载 Google Drive**。

In [ ]:
# Cell 1：安装依赖（每次新 Session 都需执行，约 3-5 分钟）
# 安装完成后 Colab 可能提示重启内核——点击「Restart」后从 Cell 2 继续即可
!pip install unsloth trl peft bitsandbytes transformers datasets accelerate pyyaml safetensors -q

import unsloth, trl, transformers, peft
print(f"✅ Unsloth {unsloth.__version__} | TRL {trl.__version__} "
      f"| Transformers {transformers.__version__} | PEFT {peft.__version__}")

In [ ]:
# Cell 2：挂载 Google Drive（可选，强烈推荐）
# ─────────────────────────────────────────────────────────────────
# 免费 Colab 会话结束后 /content 下所有数据丢失！
# 挂载 Drive 后训练产物将持久保存，下次 Session 可直接续跑。
# ─────────────────────────────────────────────────────────────────
MOUNT_DRIVE = False  # ← 改为 True 以挂载 Google Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/Qwen-Reasoning/outputs'
    import os; os.makedirs(OUTPUT_ROOT, exist_ok=True)
    print(f"✅ Google Drive 已挂载")
    print(f"   训练产物将保存至: {OUTPUT_ROOT}")
else:
    OUTPUT_ROOT = '/content/6000Q-QwenMiniReason/outputs'
    print("⚠️  未挂载 Google Drive，产物保存在 /content（会话结束后丢失）")
    print("   如需持久化，请将 MOUNT_DRIVE 改为 True 后重新运行此 Cell")

print(f"\nOUTPUT_ROOT = {OUTPUT_ROOT}")

In [ ]:
# Cell 3：克隆项目 & 创建目录
import os, subprocess

REPO_URL = "https://github.com/yukiiii0730/6000Q-QwenMiniReason.git"
REPO_DIR = "/content/6000Q-QwenMiniReason"

if not os.path.exists(os.path.join(REPO_DIR, "config", "sft_config.yaml")):
    print(f"📥 克隆项目: {REPO_URL}")
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ 克隆失败，请检查网络或 REPO_URL\n{result.stderr}")
    else:
        print("✅ 克隆完成")
else:
    print(f"✅ 项目目录已存在: {REPO_DIR}")
    # 拉取最新代码（可选）
    # subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)

# 设置工作目录（此后所有 !python 命令均从项目根目录执行）
os.chdir(REPO_DIR)
print(f"📂 工作目录: {os.getcwd()}")

# 创建必要目录
try:
    OUTPUT_ROOT
except NameError:
    OUTPUT_ROOT = "outputs"

for d in [f"{OUTPUT_ROOT}/sft", f"{OUTPUT_ROOT}/dpo",
          f"{OUTPUT_ROOT}/merged", f"{OUTPUT_ROOT}/sft_merged",
          f"{OUTPUT_ROOT}/dpo_merged", "data/processed", "logs"]:
    os.makedirs(d, exist_ok=True)
print("✅ 目录结构已创建")

In [ ]:
# Cell 4：配置 HuggingFace Token
# ─────────────────────────────────────────────────────────────────
# 推荐：在 Colab 左侧「🔑 Secrets」面板添加名为 HF_TOKEN 的 Secret
# 访问公开数据集（本项目用的数据集均公开）可以不填 Token
# ─────────────────────────────────────────────────────────────────
import os, yaml

HF_TOKEN = ""

# 优先：Colab Secrets（最安全）
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
    if HF_TOKEN:
        print("✅ 已从 Colab Secrets (HF_TOKEN) 读取 Token")
except Exception:
    pass

# 备选：config yaml
if not HF_TOKEN:
    try:
        with open("config/sft_config.yaml") as f:
            t = yaml.safe_load(f).get("hf_token", "").strip()
        if t and t not in ("YOUR_HF_TOKEN_HERE", ""):
            HF_TOKEN = t
            print("✅ 已从 config/sft_config.yaml 读取 Token")
    except Exception:
        pass

# 备选：环境变量
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
    if HF_TOKEN:
        print("✅ 已从环境变量 HF_TOKEN 读取 Token")

if not HF_TOKEN:
    print("⚠️  未找到 HuggingFace Token（使用公开数据集可以继续）")
    print("   建议：Colab 左侧「🔑 Secrets」添加名为 HF_TOKEN 的 Secret")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    # 同步写入 config yaml，供训练脚本读取
    for cfg_path in ["config/sft_config.yaml", "config/dpo_config.yaml"]:
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        cfg["hf_token"] = HF_TOKEN
        with open(cfg_path, "w") as f:
            yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)
    print("✅ Token 已同步至 config yaml")

In [ ]:
# Cell 5：GPU 检查 & 自动适配超参数
import torch, yaml, os

try:
    OUTPUT_ROOT
except NameError:
    OUTPUT_ROOT = "outputs"

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU（无 GPU）"
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
bf16_ok  = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(f"GPU  : {gpu_name}")
print(f"显存 : {vram_gb} GB")
print(f"BF16 : {'✅ 支持' if bf16_ok else '❌ 不支持（使用 FP16）'}")

# ─────────────────────────────────────────────────────────────────
# 根据显存自动选择采样量和训练步数
#   A100/H100  (≥ 40 GB) : 5 万条，充分训练
#   中高显存   (≥ 15 GB) : 2 万条，均衡训练
#   T4 / 免费  (<  15 GB): 5 千条，轻量训练
# 如需快速验证管道，可在下方手动改小 SFT_STEPS / DPO_STEPS
# ─────────────────────────────────────────────────────────────────
if vram_gb >= 40:
    SFT_SAMPLES, DPO_SAMPLES = 50000, 50000
    SFT_STEPS,   DPO_STEPS   = 1000,  500
    EVAL_N  = 200
    preset  = "A100 全量训练"
elif vram_gb >= 15:
    SFT_SAMPLES, DPO_SAMPLES = 20000, 20000
    SFT_STEPS,   DPO_STEPS   = 500,   200
    EVAL_N  = 100
    preset  = "中高显存训练"
else:
    SFT_SAMPLES, DPO_SAMPLES = 5000, 5000
    SFT_STEPS,   DPO_STEPS   = 200,  100
    EVAL_N  = 50
    preset  = "T4 轻量训练"

# ── 手动微调（快速验证管道时取消注释） ───────────────────────────
# SFT_STEPS, DPO_STEPS, EVAL_N = 50, 30, 50

print(f"\n⚙️  预设方案：{preset}")
print(f"   SFT: 每数据集取 {SFT_SAMPLES:,} 条，max_steps={SFT_STEPS}")
print(f"   DPO: 取 {DPO_SAMPLES:,} 条，max_steps={DPO_STEPS}")
print(f"   Eval: 前 {EVAL_N} 条")

# ── 写入 SFT config ───────────────────────────────────────────────
with open("config/sft_config.yaml") as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg["output_dir"]         = f"{OUTPUT_ROOT}/sft"
sft_cfg["train"]["bf16"]      = bf16_ok
sft_cfg["train"]["fp16"]      = not bf16_ok
sft_cfg["train"]["max_steps"] = SFT_STEPS
sft_cfg.pop("datasets", None)   # Colab 模式用本地 JSON
with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True, sort_keys=False)

# ── 写入 DPO config ───────────────────────────────────────────────
with open("config/dpo_config.yaml") as f:
    dpo_cfg = yaml.safe_load(f)
dpo_cfg["output_dir"]          = f"{OUTPUT_ROOT}/dpo"
dpo_cfg["base_adapter_path"]   = f"{OUTPUT_ROOT}/sft"
dpo_cfg["train"]["bf16"]       = bf16_ok
dpo_cfg["train"]["fp16"]       = not bf16_ok
dpo_cfg["train"]["max_steps"]  = DPO_STEPS
dpo_cfg.pop("dataset", None)    # Colab 模式用本地 JSON
with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True, sort_keys=False)

for d in [f"{OUTPUT_ROOT}/sft", f"{OUTPUT_ROOT}/dpo",
          f"{OUTPUT_ROOT}/merged", "data/processed", "logs"]:
    os.makedirs(d, exist_ok=True)

print("\n✅ config yaml 已更新，输出目录已创建")

In [ ]:
# Cell 6：数据集下载 & 本地 JSON 缓存
# 将 HuggingFace 数据集转换为本地 JSON，后续重跑时直接读取缓存（跳过下载）
import json, os, yaml
from datasets import load_dataset

PROCESSED = "data/processed"
SFT_OUT   = f"{PROCESSED}/sft_train.json"
DPO_OUT   = f"{PROCESSED}/dpo_train.json"
DEFAULT_INSTRUCTION = "请先进行清晰的逐步推理，再给出最终答案。"

try:
    SFT_SAMPLES, DPO_SAMPLES
except NameError:
    SFT_SAMPLES, DPO_SAMPLES = 5000, 5000
    print("⚠️  未检测到 GPU check 变量，使用默认 5000 条（请先运行 Cell 5）")

# ── SFT 数据 ──────────────────────────────────────────────────────
if os.path.exists(SFT_OUT) and os.path.getsize(SFT_OUT) > 100:
    print(f"⏭️  SFT 缓存已存在: {SFT_OUT}")
else:
    rows = []
    print(f"📥 NuminaMath-CoT（前 {SFT_SAMPLES:,} 条）...")
    numina = load_dataset("AI-MO/NuminaMath-CoT", split="train")
    for x in numina.select(range(min(SFT_SAMPLES, len(numina)))):
        rows.append({"instruction": DEFAULT_INSTRUCTION,
                     "input":  x["problem"].strip(),
                     "output": x["solution"].strip()})

    print(f"📥 Magpie-Reasoning-150K（前 {SFT_SAMPLES:,} 条）...")
    magpie = load_dataset("Magpie-Align/Magpie-Reasoning-150K", split="train")
    for x in magpie.select(range(min(SFT_SAMPLES, len(magpie)))):
        rows.append({"instruction": x["instruction"].strip(),
                     "input":  "",
                     "output": x["response"].strip()})

    with open(SFT_OUT, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"✅ SFT: {len(rows):,} 条 → {SFT_OUT}")

# ── DPO 数据 ──────────────────────────────────────────────────────
if os.path.exists(DPO_OUT) and os.path.getsize(DPO_OUT) > 100:
    print(f"⏭️  DPO 缓存已存在: {DPO_OUT}")
else:
    print(f"📥 orca-math-word-problems-200k（前 {DPO_SAMPLES:,} 条）...")
    orca = load_dataset("microsoft/orca-math-word-problems-200k", split="train")
    rows = []
    for x in orca.select(range(min(DPO_SAMPLES, len(orca)))):
        rej = x.get("incorrect_solution", "")
        if not rej:
            continue
        rows.append({"prompt":   x["question"].strip(),
                     "chosen":   x.get("correct_solution", "").strip(),
                     "rejected": rej.strip()})
    with open(DPO_OUT, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"✅ DPO: {len(rows):,} 条 → {DPO_OUT}")

# ── 将本地路径写回 config ─────────────────────────────────────────
for cfg_path, out_path in [("config/sft_config.yaml", SFT_OUT),
                            ("config/dpo_config.yaml", DPO_OUT)]:
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    cfg["dataset_path"] = out_path
    cfg.pop("datasets", None)
    cfg.pop("dataset",  None)
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print("\n✅ config 已更新 → dataset_path 指向本地缓存")

In [ ]:
# Cell 7（可选）：Baseline GSM8K 评测
# 用于记录微调前的原始模型性能，便于最后对比；跳过此 Cell 不影响训练
# ─────────────────────────────────────────────────────────────────
import re, json, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

BASELINE_OUT = "logs/gsm8k_baseline.json"
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
try:
    EVAL_N
except NameError:
    EVAL_N = 50

if os.path.exists(BASELINE_OUT):
    print(f"⏭️  Baseline 已存在: {BASELINE_OUT}")
    with open(BASELINE_OUT) as f:
        r = json.load(f)
    print(f"   GSM8K Baseline: {r['accuracy']:.2%}  ({r['correct']}/{r['total']})")
else:
    def _extract_num(t):
        m = re.findall(r"-?\d+(?:\.\d+)?", t.replace(",", ""))
        return m[-1] if m else ""

    print(f"📥 加载 {BASE_MODEL} ...")
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    mdl = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, device_map="auto", torch_dtype=torch.float16)

    gsm     = load_dataset("gsm8k", "main", split="test").select(range(EVAL_N))
    correct, details = 0, []
    for i, ex in enumerate(gsm):
        prompt  = f"请解答以下数学题，逐步推理后在最后给出数字答案。\n题目：{ex['question']}\n解答："
        inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
        out     = mdl.generate(**inputs, max_new_tokens=256, do_sample=False)
        text    = tok.decode(out[0], skip_special_tokens=True)
        pred    = _extract_num(text)
        gt      = _extract_num(ex["answer"])
        ok      = pred == gt and pred != ""
        correct += int(ok)
        details.append({"pred": pred, "gt": gt, "correct": ok})
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{EVAL_N}  acc={correct/(i+1):.2%}")

    acc    = correct / max(len(details), 1)
    result = {"model": BASE_MODEL, "accuracy": acc,
              "total": len(details), "correct": correct}
    os.makedirs("logs", exist_ok=True)
    with open(BASELINE_OUT, "w") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Baseline GSM8K: {acc:.2%}  ({correct}/{len(details)})")

    # 释放显存，为训练腾出空间
    del mdl, tok
    torch.cuda.empty_cache()
    print("🧹 GPU 显存已释放，可继续训练")

In [ ]:
# Cell 8：SFT 监督微调
# 日志实时输出，完成后产物保存至 config 中的 output_dir（默认 outputs/sft）
!python scripts/sft_train.py --config config/sft_config.yaml

In [ ]:
# Cell 9：验证 SFT 产物
import os, yaml

with open("config/sft_config.yaml") as f:
    sft_dir = yaml.safe_load(f)["output_dir"]

if os.path.exists(sft_dir):
    files = os.listdir(sft_dir)
    if any(f in ("adapter_config.json",) or f.endswith((".bin", ".safetensors"))
           for f in files):
        print(f"✅ SFT 产物已保存: {sft_dir}")
        print("   文件:", files)
    else:
        print(f"❌ 目录存在但未找到模型文件，请检查训练日志")
        print("   当前内容:", files)
else:
    print(f"❌ 输出目录不存在: {sft_dir}，请检查 SFT 训练是否成功")

In [ ]:
# Cell 10：DPO 偏好优化
# 基于 SFT adapter 继续训练，日志实时输出
!python scripts/dpo_train.py --config config/dpo_config.yaml

In [ ]:
# Cell 11：合并 LoRA 权重
# 将 DPO adapter（SFT+DPO 两阶段）合并回 fp16 基础模型，用于推理和评测
import os, yaml, subprocess

with open("config/dpo_config.yaml") as f:
    cfg = yaml.safe_load(f)

dpo_dir     = cfg["output_dir"]                        # e.g. outputs/dpo
output_root = os.path.dirname(dpo_dir)                 # e.g. outputs
merged_dir  = os.path.join(output_root, "merged")

print(f"🔗 合并: {dpo_dir} → {merged_dir}")
result = subprocess.run(
    ["python", "scripts/merge_lora.py",
     "--adapter_path", dpo_dir,
     "--output_path",  merged_dir],
    text=True
)
if result.returncode == 0:
    print(f"✅ 权重合并完成 → {merged_dir}")
    print("   文件:", os.listdir(merged_dir))
else:
    print("❌ 合并失败，请检查上方日志")

In [ ]:
# Cell 12：推理对比——基础模型 vs 微调模型（SFT+DPO）
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os, yaml

with open("config/dpo_config.yaml") as f:
    cfg = yaml.safe_load(f)
merged_dir = os.path.join(os.path.dirname(cfg["output_dir"]), "merged")

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
FT_MODEL   = merged_dir
QUESTION   = "小明有12个苹果，给了小红3个，又买了5个，现在有几个？请一步步推理后给出答案。"

def infer(model_path, prompt, max_new_tokens=512):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto",
        torch_dtype=torch.float16, trust_remote_code=True)
    inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text    = tok.decode(outputs[0], skip_special_tokens=True)
    del mdl, tok; torch.cuda.empty_cache()
    return text

print("=" * 60)
print("【基础模型 (Baseline)】")
print(infer(BASE_MODEL, QUESTION))
print()
print("=" * 60)
print("【微调模型 (SFT + DPO)】")
print(infer(FT_MODEL, QUESTION))

In [ ]:
# Cell 13：最终 GSM8K 评测 & 对比表
import subprocess, json, os, yaml

with open("config/dpo_config.yaml") as f:
    cfg = yaml.safe_load(f)
merged_dir = os.path.join(os.path.dirname(cfg["output_dir"]), "merged")

try:
    EVAL_N
except NameError:
    EVAL_N = 50

RESULT_OUT = "logs/gsm8k_result.json"

if os.path.exists(RESULT_OUT):
    print(f"⏭️  评测结果已存在: {RESULT_OUT}（删除文件后可重新评测）")
else:
    print(f"📊 GSM8K 评测（前 {EVAL_N} 条）→ {merged_dir}")
    subprocess.run([
        "python", "eval/gsm8k_eval.py",
        "--model_path",  merged_dir,
        "--max_samples", str(EVAL_N),
        "--output",      RESULT_OUT,
    ])

# ── 对比表 ────────────────────────────────────────────────────────
def load_r(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

base = load_r("logs/gsm8k_baseline.json")
ft   = load_r(RESULT_OUT)

def fmt(r):
    if r is None: return "  N/A  "
    return f"{r['accuracy']:>7.2%}  ({r['correct']}/{r['total']})"

def delta_fmt(r, b):
    if r is None or b is None: return "  N/A  "
    return f"{r['accuracy'] - b['accuracy']:>+7.2%}"

print()
print("=" * 60)
print(f"  {'模型':<30}  {'GSM8K':>9}")
print("=" * 60)
print(f"  {'Baseline (原始模型)':<30}  {fmt(base)}")
print(f"  {'SFT + DPO (微调后)':<30}  {fmt(ft)}")
print("-" * 60)
print(f"  {'Δ vs Baseline':<30}  {delta_fmt(ft, base)}")
print("=" * 60)

# 保存汇总
metrics = {}
if base: metrics["baseline_gsm8k"] = base["accuracy"]
if ft:   metrics["finetuned_gsm8k"] = ft["accuracy"]
with open("logs/compare_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ 汇总结果已保存至 logs/compare_metrics.json")